# **Rolling Stock ETL**

### Data Fetching

In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        query = f"SELECT * FROM {table_name};"
        
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_18840\66124713.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 6834 rows from 'extraction'


In [2]:
df = df_original.copy(deep=True)
df = df[(df['status_id'] == 1) & (df['dept_name'] == 'Rolling-Stock')][['filename', 'workorder_id', 'json_data']]

df.head(3)

len(df)

1206

In [3]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['air_standup', 'airbag-pressure', 'airbag_pressure', 'approval', 'cardan_shaft', 'cceb', 'greasing_cardan_shaft', 'notification', 'stamping', 'technician', 'train_startup_test', 'tyre-pressure', 'tyre-wear', 'tyre_pressure', 'tyre_wear', 'water_ponding', 'work_order']


### Cardan Shaft

In [4]:
df['cardan_shaft'] = df['json_data'].apply(
    lambda x: x.get('cardan_shaft') if isinstance(x, dict) else None
)

df['workorder_id'] = (
    df['workorder_id']
    .apply(lambda x: int(x) if pd.notnull(x) else None)
)

df[['filename', 'workorder_id', 'cardan_shaft']].head(1).to_dict(orient='records')

[{'filename': 'RS_PM_WEK_4000586856.pdf',
  'workorder_id': 4000586856,
  'cardan_shaft': {'date': '25/02/2024',
   'technician_id': '19580,20674',
   'supervisor_id': 7127,
   'bogie_checks': {'bogie2': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}},
    'bogie3': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}},
    'bogie4': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}},
    'bogie5': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}},
    'bogie6': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}},
    'bogie7': {'long_cardan_shaft': {'a': True, 'b': True},
     'short_cardan_shaft': {'c': True, 'd': True}}}}}]

In [5]:
import pandas as pd
import numpy as np

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['cardan_shaft'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

df_with_na[['filename', 'workorder_id', 'na_keys']].head(5)

,filename,workorder_id,na_keys
11,RS_PM_MTH_4000460878.pdf,4000460878,[technician_id]
83,RS_PM_MTH_4000553634.pdf,4000553634,"[technician_id, supervisor_id]"
94,RS_PM_MTH_4000578099.pdf,4000578099,[technician_id]
118,RS_PM_MTH_4000654565.pdf,4000654565,"[technician_id, supervisor_id]"
120,RS_PM_MTH_4000651771.pdf,4000651771,[supervisor_id]


In [6]:
from collections import Counter

na_counter = Counter(k for keys in df['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

print(na_summary)

             key  na_count
1  supervisor_id        26
0  technician_id        24
2           date         6


In [7]:

import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['cardan_shaft'] = df['cardan_shaft'].apply(clean_value)

df['cardan_shaft'] = df['cardan_shaft'].replace(np.nan, '', regex=True)

df['cardan_shaft'].head(3)

0    {'date': '25/02/2024', 'technician_id': '19580...
1    {'date': '12/05/2022', 'technician_id': 7232, ...
3    {'date': '25/01/2022', 'technician_id': '11139...
Name: cardan_shaft, dtype: object

In [8]:
def extract_leaf_keys(d, parent=''):
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

df['cardan_shaft_leaf_keys'] = df['cardan_shaft'].apply(
    lambda x: extract_leaf_keys(x) if isinstance(x, dict) else []
)

unique_keys = sorted(set(k for sublist in df['cardan_shaft_leaf_keys'] for k in sublist))

for k in unique_keys:
    print(k)


bogie_checks.bogie2.long_cardan_shaft.a
bogie_checks.bogie2.long_cardan_shaft.b
bogie_checks.bogie2.short_cardan_shaft.c
bogie_checks.bogie2.short_cardan_shaft.d
bogie_checks.bogie3.long_cardan_shaft.a
bogie_checks.bogie3.long_cardan_shaft.b
bogie_checks.bogie3.short_cardan_shaft.c
bogie_checks.bogie3.short_cardan_shaft.d
bogie_checks.bogie4.long_cardan_shaft.a
bogie_checks.bogie4.long_cardan_shaft.b
bogie_checks.bogie4.short_cardan_shaft.c
bogie_checks.bogie4.short_cardan_shaft.d
bogie_checks.bogie5.long_cardan_shaft.a
bogie_checks.bogie5.long_cardan_shaft.b
bogie_checks.bogie5.short_cardan_shaft.c
bogie_checks.bogie5.short_cardan_shaft.d
bogie_checks.bogie6.long_cardan_shaft.a
bogie_checks.bogie6.long_cardan_shaft.b
bogie_checks.bogie6.short_cardan_shaft.c
bogie_checks.bogie6.short_cardan_shaft.d
bogie_checks.bogie7.long_cardan_shaft.a
bogie_checks.bogie7.long_cardan_shaft.b
bogie_checks.bogie7.short_cardan_shaft.c
bogie_checks.bogie7.short_cardan_shaft.d
date
supervisor_id
technicia

In [9]:
import json
import pandas as pd

def flatten_with_descriptions(row):
    flat = {}

    def recurse(subdict, parent=''):
        if subdict is None:
            return

        if isinstance(subdict, str):
            try:
                subdict = json.loads(subdict)
            except json.JSONDecodeError:
                return

        if not isinstance(subdict, dict):
            return

        for k, v in subdict.items():
            if len(k) == 1 and k.isalpha():
                new_parent = parent
            else:
                new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = (
                        desc.lower()
                        .replace(' ', '_')
                        .replace('/', '_')
                        .replace('&', 'and')
                    )
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(row)
    return flat

flattened_rows = [flatten_with_descriptions(r) for r in df['cardan_shaft'].fillna({})]

cardanshaft_df = pd.DataFrame(flattened_rows)
cardanshaft_df.index = df.index
cardanshaft_df['workorder_id'] = df['workorder_id'].astype('Int64')
cardanshaft_df['filename'] = df['filename']

cardanshaft_df

,date,technician_id,supervisor_id,bogie_checks.bogie2.long_cardan_shaft,bogie_checks.bogie2.short_cardan_shaft,bogie_checks.bogie3.long_cardan_shaft,bogie_checks.bogie3.short_cardan_shaft,bogie_checks.bogie4.long_cardan_shaft,bogie_checks.bogie4.short_cardan_shaft,bogie_checks.bogie5.long_cardan_shaft,bogie_checks.bogie5.short_cardan_shaft,bogie_checks.bogie6.long_cardan_shaft,bogie_checks.bogie6.short_cardan_shaft,bogie_checks.bogie7.long_cardan_shaft,bogie_checks.bogie7.short_cardan_shaft,workorder_id,filename
0,25/02/2024,"19580,20674",7127,True,True,True,True,True,True,True,True,True,True,True,True,4000586856,RS_PM_WEK_4000586856.pdf
1,12/05/2022,7232,7066,True,True,True,True,True,True,True,True,True,True,True,True,4000464193,RS_PM_MTH_4000464193.pdf
3,25/01/2022,11139,7127,True,True,True,True,True,True,True,True,True,True,True,True,4000446287,RS_PM_MTH_4000446287.pdf
4,09/10/2023,11083,7192,True,True,True,True,True,True,True,True,True,True,True,True,4000558454,RS_PM_WEK_4000558454.pdf
5,17/05/2022,MSF,7127,True,True,True,True,True,True,True,True,True,True,True,True,4000464732,RS_PM_MTH_4000464732.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000614023,RS_PM_QTR_4000614023.pdf
6715,12/12/2022,10018745,7196,True,True,True,True,True,True,True,True,True,True,True,True,4000501707,RS_PM_MTH_4000501707.pdf
6717,07/02/2023,9435,7196,True,True,True,True,True,True,True,True,True,True,True,True,4000522314,RS_PM_WEK_4000522314.pdf
6759,08/09/2024,19791,7192,True,True,True,True,True,True,True,True,True,True,True,True,4000625781,RS_PM_WEK_4000625781.pdf


In [10]:
output_path = '../../output/rolling_stock.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    cardanshaft_df.to_excel(writer, index=False, sheet_name='cardan_shaft')

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")

✅ Exported successfully to '../../output/rolling_stock.xlsx' (replaced existing sheet)
